# 06. Spectroscopy (IR / UV-Vis) & Powder X-Ray Diffraction

`xkcd-matchem` parses harmonic IR frequencies from Gaussian `.log`, TDDFT UV-Vis transitions from ORCA `.out`, and computes simulated powder XRD patterns from any periodic crystal using pure-NumPy reciprocal-lattice enumeration and **Cromer-Mann (LA-3816)** atomic scattering form factors (plus the exact analytical hydrogen $1s$ factor) at $\text{Cu K}\alpha_1$ (`1.540598` Å per CODATA 2022).

:::{note}
xkcd-style spectra are great for teaching peak assignments, comparing fingerprints in group meetings, and illustrating concepts on posters. For archival Rietveld refinements or quantitative peak-shape fitting, always use standard straight-line Matplotlib plots.
:::

## Basic Usage

Let's sketch water's IR modes (`h2o_gaussian.log`), benzene's UV-Vis transitions (`benzene_orca.out`), and diamond silicon's powder diffraction peaks (`si_diamond.cif`) side by side:

In [ ]:
import matplotlib.pyplot as plt
import xkcd_matchem as xm

h2o = xm.load("h2o_gaussian.log")
benz = xm.load("benzene_orca.out")

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3))
    xm.plot_spectrum(h2o.metadata["ir_spectrum"], ax=axes[0], title="H2O IR (Gaussian)")
    xm.plot_spectrum(benz.metadata["uvvis_spectrum"], ax=axes[1], title="Benzene UV-Vis (ORCA)")
    xm.plot_xrd("si_diamond.cif", two_theta_range=(20.0, 85.0), ax=axes[2], title="Si Cu Ka1 Powder XRD")
    fig.tight_layout()

## Advanced Usage

Want to inspect the individual $(hkl)$ reflections or swap the broadening lineshape? Call `xm.compute_xrd()` to inspect `Spectrum.peaks`, then broaden them onto any custom $2\theta$ grid with `xm.broaden_peaks(..., profile="pseudo_voigt")` (also supports `"gaussian"` and `"lorentzian"`):

In [ ]:
import numpy as np

nacl_spec = xm.load("nacl.cif")
xrd_spec = xm.compute_xrd(nacl_spec, two_theta_range=(15.0, 75.0), fwhm=0.55)
top_reflections = [
    (round(p.position, 2), round(p.intensity, 1), p.label)
    for p in sorted(xrd_spec.peaks, key=lambda k: k.intensity, reverse=True)[:3]
]
print("Top 3 reflections (2theta, rel. intensity, hkl):", top_reflections)

custom_grid = np.linspace(15.0, 75.0, 800)
pv_curve = xm.broaden_peaks(
    [p.position for p in xrd_spec.peaks],
    [p.intensity for p in xrd_spec.peaks],
    custom_grid,
    fwhm=0.8,
    profile="pseudo_voigt",
)
pv_spec = xm.Spectrum(
    x=custom_grid,
    y=pv_curve / max(pv_curve.max(), 1e-6) * 100.0,
    kind="xrd",
    x_label="2theta (deg)",
    y_label="Normalized Intensity (%)",
    peaks=xrd_spec.peaks,
)
ax = xm.plot_xrd(
    pv_spec,
    two_theta_range=(15.0, 75.0),
    label_peaks=4,
    color="#882255",
    title="Custom Pseudo-Voigt NaCl XRD",
)

:::{tip}
**Try Next:** Pass `wavelength=0.71073` to `compute_xrd()` or `plot_xrd()` to switch from $\text{Cu K}\alpha_1$ to $\text{Mo K}\alpha$ radiation, or continue to **Tutorial 07** for reaction energy profiles and NEB transition-state barriers.
:::